# Generar máscaras propuestas de pares (issue #9)

Para cada **par** (un encuadre en T1 = antes y T2 = después, ADR-0011) arma lo que se valida en QGIS:
- `<id>_propuesta.gpkg`: máscara propuesta, con SAM sobre T2 y un clic en cada locación nueva del padrón,
- `<id>_dndvi.tif`: caída de NDVI entre T1 y T2, la capa de ayuda para ver qué cambió.

**Mientras no estén los pares de Sofi (#11)**, este notebook arma **10 pares de prueba** con imágenes de
Planetary Computer, siguiendo la regla de METODO.md: T1 = mes del primer pozo − 6 y T2 = mes del primer pozo + 1.
Sirven para el #10 (medir tiempos y contar en cuántos la locación ya se veía en T1).
Cuando estén los pares oficiales, se salta la sección 2 y se apunta `CARPETA` a ellos.

**Antes de correr:** Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU**.

In [ ]:
RAMA = "9-generador-mascaras"
!git clone -q -b {RAMA} https://github.com/sofia2005222/vacamuerta-metano-seg.git
%cd vacamuerta-metano-seg
!pip install -qq -e ".[dev]" pystac-client planetary-computer odc-stac segment-anything scipy
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
import torch; print("GPU:", torch.cuda.is_available())

## 1. Parámetros

- `N_PARES`: cuántos pares de prueba (el #10 pide 10).
- `CONTEXTO`: lado en píxeles de la imagen T2 que ve SAM (512 = como la variante A del #8, que dio 20/20).

In [ ]:
N_PARES = 10
SEMILLA = 0
CARPETA = "pares_prueba"
CRS = "EPSG:32719"
CONTEXTO = 512
TAMANO = 256

## 2. Pares de prueba desde Planetary Computer

Se eligen locaciones nuevas (primer pozo entre julio de 2019 y diciembre de 2023) de la Zona Añelo, repartidas en el
tiempo. Para cada una se baja la escena con menos nubes de T1 y de T2 en un cuadrado de 5,12 km, se guarda el de T2
entero como contexto para SAM y se corta el recorte de 256 px con la locación en una posición al azar (no siempre
en el centro, como en los encuadres reales). La máscara queda en 255 (sin etiqueta) hasta validarla.

> Diferencia con los pares oficiales: acá cada fecha es **una escena** (la más limpia del mes), no la mediana mensual de GEE.

In [ ]:
import os, numpy as np, pandas as pd, geopandas as gpd, rasterio
import pystac_client, planetary_computer, odc.stac
from pyproj import Transformer
from vmseg.par import guardar_par, guardar_tif, COLUMNAS_INDICE_PARES
from vmseg.recorte import SIN_ETIQUETA

catalogo = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                     modifier=planetary_computer.sign_inplace)
BANDAS_PC = ["B02", "B03", "B04", "B08", "B11", "B12"]
a_lonlat = Transformer.from_crs(CRS, "EPSG:4326", always_xy=True)

def bajar(mes, x_min, y_max, lado):
    """Escena con menos nubes del mes, en un cuadrado de `lado` px (10 m) con esquina (x_min, y_max)."""
    x_max, y_min = x_min + lado * 10, y_max - lado * 10
    (lon0, lat0), (lon1, lat1) = a_lonlat.transform(x_min, y_min), a_lonlat.transform(x_max, y_max)
    p = pd.Period(mes, freq="M")
    items = list(catalogo.search(collections=["sentinel-2-l2a"], bbox=[lon0, lat0, lon1, lat1],
                                 datetime=f"{p.start_time:%Y-%m-%d}/{p.end_time:%Y-%m-%d}",
                                 query={"eo:cloud_cover": {"lt": 20}}).items())
    if not items:
        return None
    mejor = min(items, key=lambda it: it.properties["eo:cloud_cover"])
    del_dia = [it for it in items if it.datetime.date() == mejor.datetime.date()]
    ds = odc.stac.load(del_dia, bands=BANDAS_PC, crs=CRS, resolution=10, resampling="bilinear",
                       groupby="solar_day", x=(x_min, x_max), y=(y_min, y_max))
    img = ds[BANDAS_PC].to_array().squeeze("time").values.astype("float32")[:, :lado, :lado]
    if mejor.properties.get("s2:processing_baseline", "00.00") >= "04.00":
        img = img - 1000
    return np.clip(img / 10000, 0, 1)

padron = gpd.read_file("data/locaciones_vaca_muerta.geojson")
loc = padron[padron.fecha_primer_pozo.notna()].copy()
loc["fecha"] = pd.to_datetime(loc.fecha_primer_pozo)
nuevas = loc[(loc.fecha >= "2019-07-01") & (loc.fecha <= "2023-12-31")
             & (loc.en_zona_anelo.astype(str) == "True")].sort_values("fecha")
elegidas = nuevas.iloc[np.linspace(0, len(nuevas) - 1, N_PARES).round().astype(int)].to_crs(CRS)

rng = np.random.default_rng(SEMILLA)
os.makedirs(f"{CARPETA}/contexto", exist_ok=True)
filas = []
for f in elegidas.itertuples():
    mes = pd.Period(f.fecha, freq="M")
    mes_t1, mes_t2 = str(mes - 6), str(mes + 1)
    # contexto de 512 px centrado en la locación; el recorte de 256 px, corrido al azar
    cx, cy = round(f.geometry.x / 10) * 10, round(f.geometry.y / 10) * 10
    x_ctx, y_ctx = cx - CONTEXTO * 5, cy + CONTEXTO * 5
    df, dc = rng.integers(-80, 81, size=2)
    f0, c0 = (CONTEXTO - TAMANO) // 2 + df, (CONTEXTO - TAMANO) // 2 + dc
    t1, t2 = bajar(mes_t1, x_ctx, y_ctx, CONTEXTO), bajar(mes_t2, x_ctx, y_ctx, CONTEXTO)
    if t1 is None or t2 is None:
        print(f.locacion_id, ": sin escena limpia en", mes_t1 if t1 is None else mes_t2); continue
    id_par = f"prueba_{f.locacion_id}"
    guardar_tif(f"{CARPETA}/contexto/{id_par}_t2_contexto.tif", t2, x_ctx, y_ctx, CRS, "float32")
    x_min, y_max = x_ctx + c0 * 10, y_ctx - f0 * 10
    guardar_par(CARPETA, id_par, t1[:, f0:f0 + TAMANO, c0:c0 + TAMANO], t2[:, f0:f0 + TAMANO, c0:c0 + TAMANO],
                np.full((TAMANO, TAMANO), SIN_ETIQUETA, np.uint8), x_min, y_max)
    filas.append(dict(id=id_par, encuadre="prueba", locacion_id=int(f.locacion_id), zona="anelo",
                      bloque=int(f.bloque_anelo) if pd.notna(f.bloque_anelo) else -1, split=f.split,
                      mes_t1=mes_t1, mes_t2=mes_t2, x_min=x_min, y_max=y_max, crs=CRS,
                      validada=False, nota=f"primer pozo {f.fecha:%Y-%m-%d}"))
    print(id_par, mes_t1, "→", mes_t2)
indice = pd.DataFrame(filas, columns=COLUMNAS_INDICE_PARES)
indice.to_csv(f"{CARPETA}/indice.csv", index=False)
indice

## 3. SAM sobre T2 y capa de caída de NDVI

In [ ]:
from segment_anything import sam_model_registry, SamPredictor
from vmseg.mascaras_propuestas import proponer_par, segmentador_sam

sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth").to("cuda" if torch.cuda.is_available() else "cpu")
segmentar = segmentador_sam(SamPredictor(sam))

propuestas = {}
for fila in indice.itertuples():
    contexto = f"{CARPETA}/contexto/{fila.id}_t2_contexto.tif"
    prop = proponer_par(CARPETA, fila.id, padron, segmentar,
                        imagen_mensual_t2=contexto if os.path.exists(contexto) else None)
    propuestas[fila.id] = prop
    print(fila.id, "→", len(prop), "polígonos")

## 4. Vista previa

Por par: T1, T2 con la propuesta (rojo; amarillo = verificar) y el punto del padrón (blanco), y la caída de NDVI.
Fijate si la locación **ya se ve en T1**: eso es lo que el #10 pide contar.

In [ ]:
import matplotlib.pyplot as plt
from vmseg.par import leer_par
from vmseg.mascaras_propuestas import rgb8

pad = padron.to_crs(CRS).set_index("locacion_id")
fig, ejes = plt.subplots(len(indice), 3, figsize=(12, 4 * len(indice)))
ejes = np.atleast_2d(ejes)
for (a, b, c), fila in zip(ejes, indice.itertuples()):
    t1, t2, _ = leer_par(CARPETA, fila.id)
    ext = [fila.x_min, fila.x_min + TAMANO * 10, fila.y_max - TAMANO * 10, fila.y_max]
    with rasterio.open(f"{CARPETA}/{fila.id}_dndvi.tif") as src:
        dndvi = src.read(1)
    a.imshow(rgb8(t1), extent=ext); a.set_title(f"{fila.id} · T1 {fila.mes_t1}", fontsize=9)
    b.imshow(rgb8(t2), extent=ext); b.set_title(f"T2 {fila.mes_t2} + propuesta", fontsize=9)
    prop = propuestas[fila.id]
    for v, color in [(0, "red"), (1, "yellow")]:
        if len(prop) and (prop.verificar == v).any():
            prop[prop.verificar == v].boundary.plot(ax=b, color=color, linewidth=1)
    p = pad.loc[fila.locacion_id].geometry
    for ax in (a, b):
        ax.plot(p.x, p.y, "o", color="white", markersize=4)
    im = c.imshow(dndvi, extent=ext, cmap="gray", vmin=0, vmax=0.15); c.set_title("caída de NDVI", fontsize=9)
    for ax in (a, b, c):
        ax.set_xlim(ext[:2]); ax.set_ylim(ext[2:]); ax.axis("off")
plt.tight_layout(); plt.savefig("pares_prueba.png", dpi=100); plt.show()

## 5. ¿Ayuda la caída de NDVI? (sugerencia del profesor)

Para cada par se compara la caída de NDVI **adentro de la propuesta** con la de un **anillo alrededor** (150–300 m).
Si adentro es claramente mayor, el NDVI marca bien dónde se desmontó. En la estepa hay poca vegetación, así que
la diferencia puede ser chica: por eso se mide.

In [ ]:
from rasterio.features import rasterize
from scipy import ndimage

res = []
for fila in indice.itertuples():
    prop = propuestas[fila.id]
    with rasterio.open(f"{CARPETA}/{fila.id}_dndvi.tif") as src:
        dndvi, t = src.read(1), src.transform
    if not len(prop):
        res.append((fila.id, np.nan, np.nan)); continue
    adentro = rasterize(prop.geometry, out_shape=dndvi.shape, transform=t).astype(bool)
    lejos = ndimage.binary_dilation(adentro, iterations=30) & ~ndimage.binary_dilation(adentro, iterations=15)
    res.append((fila.id, float(np.nanmean(dndvi[adentro])), float(np.nanmean(dndvi[lejos]))))
ndvi_res = pd.DataFrame(res, columns=["id", "caida_adentro", "caida_anillo"])
ndvi_res["diferencia"] = ndvi_res.caida_adentro - ndvi_res.caida_anillo
print(ndvi_res.round(3)); print("mediana de la diferencia:", round(ndvi_res.diferencia.median(), 3))

## 6. Planilla para el #10 y descarga

`tiempos_10.csv` queda para completar mientras validás en QGIS: minutos por par y si la locación ya se veía en T1.

Bajá `pares_prueba.zip`, descomprimilo **adentro de la carpeta del repo** (queda `vacamuerta-metano-seg/pares_prueba/`)
y seguí con QGIS: `scripts/qgis_abrir_par.py`.

In [ ]:
pd.DataFrame({"id": indice.id, "minutos": "", "se_veia_en_t1": "", "nota": ""}).to_csv(f"{CARPETA}/tiempos_10.csv", index=False)
!zip -qr pares_prueba.zip pares_prueba -x "pares_prueba/contexto/*"
from google.colab import files
for f in ["pares_prueba.zip", "pares_prueba.png"]:
    files.download(f)